In [0]:
# Silver to Gold
# 1. Aggregated Metrics
# 2. KPI's and Dashboards
# 3. ML Feature Store
# 4. Buisness Logic
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql.window import Window

# Variables
catalog = "new_databricks_workspace"
schema = "default"
# Create SparkSession
spark = SparkSession.builder.appName("Spark Dataframe").getOrCreate()

# Read Silver Info
silver_info = spark.table(f"{catalog}.{schema}.silver_athlete_information")
silver_results = spark.table(f"{catalog}.{schema}.silver_athlete_results")
silver_combined = spark.table(f"{catalog}.{schema}.silver_athlete_combined")

silver_combined.display()


In [0]:
# Gold - General Carrer Stats
# Filter out null ranks
gold_general_carrer = silver_combined.filter(F.col("rank").isNotNull()).groupBy(
    "athlete_id", "firstname", "lastname", "gender", "country",
    "height", "arm_span", "paraclimbing_sport_class", "birthday"
).agg(
    # Total competitions climbed
    f.count("*").alias("total_competitions"),
    # Total seasons climbed
    f.countDistinct("season").alias("total_seasons"),
    # Best rank
    f.min("rank").alias("best_rank"),
    # Medals
    f.sum(f.when(f.col("rank") == 1, 1).otherwise(0)).alias("gold_medals"),
    f.sum(f.when(f.col("rank") == 2, 1).otherwise(0)).alias("silver_medals"),
    f.sum(f.when(f.col("rank") == 3, 1).otherwise(0)).alias("bronze_medals"),
    f.sum(f.when(f.col("rank") <= 3, 1).otherwise(0)).alias("total_podiums"),
).withColumn(
    # Percentage of podiums
    "podium_percentage", F.round(F.col("total_podiums") / F.col("total_competitions") * 100, 2)
)
# Save results to table
gold_general_carrer.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.gold_general_career")

In [0]:

# Gold - Country Performance
# Filter out null ranks and null countries
gold_country = silver_combined.filter(
    (f.col("rank").isNotNull()) & (f.col("country").isNotNull())
).groupBy("country").agg(
    f.countDistinct("athlete_id").alias("total_athletes"),
    f.count("*").alias("total_results"),
    f.sum(f.when(f.col("rank") == 1, 1).otherwise(0)).alias("gold_medals"),
    f.sum(f.when(f.col("rank") == 2, 1).otherwise(0)).alias("silver_medals"),
    f.sum(f.when(f.col("rank") == 3, 1).otherwise(0)).alias("bronze_medals"),
    f.sum(f.when(f.col("rank") <= 3, 1).otherwise(0)).alias("total_podiums"),
    f.countDistinct("season").alias("seasons_active"),
    f.round(f.avg("rank"), 2).alias("avg_rank"),
).withColumn(
    "total_medals", f.col("gold_medals") + f.col("silver_medals") + f.col("bronze_medals")
).withColumn(
    "podium_rate", f.round(F.col("total_podiums") / f.col("total_results") * 100, 2)
)
# Save results to table
gold_country.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.gold_country_performance")